### 1. 나만의 데이터 셋 준비하기
### 2. torchvision.datasets.ImageFolder로 불러오기
### 3. transforms 적용하여 저장하기 origin_data -> train_data

In [2]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

In [3]:
from matplotlib.pyplot import imshow
%matplotlib inline

In [4]:
trans = transforms.Compose([transforms.Resize((224, 224))])
trans = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.ImageFolder(root='/Users/jooyoungson/5th-deepfashion-deep/DataSet/origin', transform = trans)

In [5]:
trans

Compose(
    ToTensor()
)

In [7]:
for num, value in enumerate(train_data):
    data, label = value
    print(num, data, label)
    
#    imshow(data)
    break

0 tensor([[[0.9765, 0.9765, 0.9765,  ..., 0.9686, 0.9686, 0.9686],
         [0.9765, 0.9765, 0.9765,  ..., 0.9725, 0.9725, 0.9725],
         [0.9804, 0.9804, 0.9804,  ..., 0.9725, 0.9725, 0.9725],
         ...,
         [0.8941, 0.8941, 0.8941,  ..., 0.9255, 0.9255, 0.9255],
         [0.8941, 0.8941, 0.8941,  ..., 0.9255, 0.9255, 0.9255],
         [0.8941, 0.8941, 0.8941,  ..., 0.9255, 0.9255, 0.9255]],

        [[0.9765, 0.9765, 0.9765,  ..., 0.9686, 0.9686, 0.9686],
         [0.9765, 0.9765, 0.9765,  ..., 0.9725, 0.9725, 0.9725],
         [0.9804, 0.9804, 0.9804,  ..., 0.9725, 0.9725, 0.9725],
         ...,
         [0.8745, 0.8745, 0.8745,  ..., 0.9176, 0.9176, 0.9176],
         [0.8745, 0.8745, 0.8745,  ..., 0.9176, 0.9176, 0.9176],
         [0.8745, 0.8745, 0.8745,  ..., 0.9176, 0.9176, 0.9176]],

        [[0.9765, 0.9765, 0.9765,  ..., 0.9765, 0.9765, 0.9765],
         [0.9765, 0.9765, 0.9765,  ..., 0.9804, 0.9804, 0.9804],
         [0.9804, 0.9804, 0.9804,  ..., 0.9804, 0.9804, 

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import torch.cuda

In [9]:
device = 'cuda' if torch.cuda.is_available() else'cpu'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [25]:
data_loader = DataLoader(dataset =  train_data, batch_size = 2, shuffle = True, num_workers = 0)

In [26]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3,6,5),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(6,16,5),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.layer3 = nn.Sequential(
            nn.Linear(16*13*29, 120),
            nn.ReLU(),
            nn.Linear(120,2)
        )
    def forward(self, x):
        out = self.layer1(x)
        print(out.shape)
        out = self.layer2(out)
        print(out.shape)
        out = out.view(out.shape[0], -1)
        print(out.shape)
        return out

In [27]:
#testing
net = CNN().to(device)
test_input = (torch.Tensor(3,3,224,224)).to(device)
test_out = net(test_input)

torch.Size([3, 6, 110, 110])
torch.Size([3, 16, 53, 53])
torch.Size([3, 44944])


In [28]:
optimizer = optim.Adam(net.parameters(), lr=0.00001)
loss_func = nn.CrossEntropyLoss().to(device)

In [29]:
total_batch = len(data_loader)

epochs = 3
for epoch in range(epochs):
    avg_cost = 0.0
    for num, data in enumerate(data_loader):
        Imgs, labels = data
        Imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out = net(imgs)
        loss = loss_func(out, labels)
        loss.backward()
        optimizer.step()
        
        avg_cost += loss / total_batch
    
    print('[Epoch:{}] cost = {}'.format(epoch+1, avg_cost))

print('Learning Finished!')

RuntimeError: invalid argument 0: Sizes of tensors must match except in dimension 0. Got 630 and 300 in dimension 2 at ../aten/src/TH/generic/THTensor.cpp:689